<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 20 - Evaluación BDA-RA1: Modelo de examen

**Módulo:** Big Data Aplicado (BDA - 5075)

**Contenido:** Ejercicios tipo examen sobre almacenamiento y procesamiento

---

Este notebook contiene ejercicios de práctica para la evaluación BDA-RA1.

## Ejercicio 1: PySpark básico

In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count

# Crear sesión
spark = SparkSession.builder.appName("Examen").master("local[*]").getOrCreate()

# Crear datos de ejemplo
data = [
    (1, "2024-01-15", "Electrónica", 1500.0, "Madrid"),
    (2, "2024-01-15", "Ropa", 200.0, "Barcelona"),
    (3, "2024-01-16", "Electrónica", 800.0, "Madrid"),
    (4, "2024-01-16", "Hogar", 350.0, "Sevilla"),
    (5, "2024-01-17", "Ropa", 150.0, "Barcelona"),
    (6, "2024-01-17", "Electrónica", 2000.0, "Valencia"),
]

df = spark.createDataFrame(data, ["id", "fecha", "categoria", "importe", "ciudad"])
print("Dataset:")
df.show()

In [ ]:
# Ejercicio: Total de ventas por categoría
print("=== VENTAS POR CATEGORÍA ===")
ventas_categoria = df.groupBy("categoria").agg(
    sum("importe").alias("total"),
    count("*").alias("transacciones"),
    avg("importe").alias("ticket_medio")
)
ventas_categoria.show()

# Ejercicio: Filtrar electrónica > 1000€
print("=== ELECTRÓNICA > 1000€ ===")
df.filter((col("categoria") == "Electrónica") & (col("importe") > 1000)).show()

## Ejercicio 2: Diseñar arquitectura Data Lake

**Pregunta:** Diseña una arquitectura Medallion para una empresa de retail.

In [ ]:
arquitectura = """
=== ARQUITECTURA MEDALLION PARA RETAIL ===

┌─────────────────────────────────────────────────────────────────┐
│                         INGESTA                                 │
│  TPV → Kafka/Kinesis → [Eventos de venta]                      │
│  ERP → Files/API    → [Inventario, productos]                  │
│  Web → Logs         → [Comportamiento usuario]                  │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    CAPA BRONZE (Raw)                           │
│  Formato: JSON/CSV tal como llega                              │
│  Particionado: fecha de ingesta                                │
│  Retención: 90 días                                            │
│  Transformación: Ninguna                                       │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    CAPA SILVER (Clean)                         │
│  Formato: Parquet                                              │
│  Transformaciones:                                             │
│    - Deduplicación                                             │
│    - Tipado correcto                                           │
│    - Validación de datos                                       │
│  Particionado: fecha del evento                                │
│  Retención: 2 años                                             │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    CAPA GOLD (Business)                        │
│  Formato: Parquet / Delta Lake                                 │
│  Tablas:                                                       │
│    - fact_ventas (particionado por mes)                        │
│    - dim_productos                                             │
│    - dim_tiendas                                               │
│    - dim_clientes                                              │
│    - agg_ventas_diarias (agregados pre-calculados)             │
│  Retención: Permanente                                         │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                       CONSUMO                                   │
│  SQL (Athena/Trino) → Dashboards                               │
│  Spark             → Modelos ML                                │
│  API               → Aplicaciones                              │
└─────────────────────────────────────────────────────────────────┘
"""

print(arquitectura)

## Ejercicio 3: Comparativa de formatos

In [ ]:
import pandas as pd

formatos = pd.DataFrame({
    'Formato': ['CSV', 'JSON', 'Parquet', 'Avro', 'ORC'],
    'Tipo': ['Texto', 'Texto', 'Columnar', 'Binario', 'Columnar'],
    'Compresión': ['No', 'No', 'Sí', 'Sí', 'Sí'],
    'Schema': ['No', 'Semi', 'Sí', 'Sí', 'Sí'],
    'Lectura parcial': ['No', 'No', 'Sí', 'No', 'Sí'],
    'Uso ideal': ['Intercambio simple', 'APIs', 'Analytics', 'Streaming', 'Hive']
})

print("=== COMPARATIVA DE FORMATOS ===")
print(formatos.to_string(index=False))

print("\n¿Por qué Parquet para Big Data?")
print("1. Columnar: Solo lee columnas necesarias")
print("2. Compresión: Reduce almacenamiento 5-10x")
print("3. Schema: Garantiza tipos de datos")
print("4. Particionable: Optimiza consultas filtradas")

In [ ]:
spark.stop()
print("\nSesión Spark cerrada")

---

## Contenidos evaluados

- Tipos de almacenamiento (SQL, NoSQL, Data Lake)
- Formatos de datos (CSV, JSON, Parquet)
- Procesamiento batch vs streaming
- Arquitecturas Cloud
- Servicios de Big Data en Cloud
- Optimización de PySpark